In [ ]:
!pip install -q neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 17.4 MB/s eta 0:00:00


In [ ]:
import neo4j

print("Neo4j Python Driver Version:", neo4j.__version__)

Neo4j Python Driver Version: 6.3.1


1.เริ่มติดต่อใช้งาน Neo4j online server

In [ ]:
from getpass import getpass
from neo4j import GraphDatabase

URI = "neo4j+s://5224144b.databases.neo4j.io"
USERNAME = "5224144b"
PASSWORD = getpass("Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully")

Password: ··········
Connected successfully


หา home database

In [ ]:
result = driver.execute_query(
    "SHOW HOME DATABASE"
)

for record in result.records:
    print(record.data())

{'name': '5224144b', 'type': 'standard', 'aliases': [], 'access': 'read-write', 'address': 'p-mt-106bc5f59e02-10-0148.production-orch-0066.neo4j.io:7687', 'role': 'primary', 'writer': True, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'constituents': []}


เก็บชื่อ database

In [ ]:
DATABASE = result.records[0]["name"]

print("Database =", DATABASE)

Database = 5224144b


หลังจากนั้น Query ทุกคำสั่งใช้:

In [ ]:
result = driver.execute_query(
    """
    RETURN 'Hello Neo4j' AS message
    """,
    database_=DATABASE
)

print(result.records[0]["message"])

Hello Neo4j


ทดสอบใช้งานฐานข้อมูลจริง

In [ ]:
result = driver.execute_query(
    """
    MATCH (n)
    RETURN count(n) AS total_nodes
    """,
    database_=DATABASE
)

print("จำนวน Node =", result.records[0]["total_nodes"])

จำนวน Node = 0


สร้าง Node

In [ ]:
result = driver.execute_query(
    """
    CREATE (u:Customer {
        customer_id: 'C001',
        name: 'Kanya'
    })

    RETURN u
    """,
    database_=DATABASE
)

print(result.records[0]["u"])

<Node element_id='4:5175a4a6-5d14-4b0e-9fa7-b8e0d311e732:0' labels=frozenset({'Customer'}) properties={'name': 'Kanya', 'customer_id': 'C001'}>


ตรวจสอบผลลัพธ์

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:Customer)

    RETURN
        u.customer_id AS id,
        u.name AS name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

{'id': 'C001', 'name': 'Kanya'}


ทดลอง where

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:Customer)

    WHERE u.customer_id = 'C001'

    RETURN
        u.name AS name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

{'name': 'Kanya'}


6.สร้าง constrain

In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT customer_id_unique
    IF NOT EXISTS

    FOR (u:Customer)

    REQUIRE u.customer_id IS UNIQUE
    """,
    database_=DATABASE
)

print("Customer constraint created")

Customer constraint created


In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT water_id_unique
    IF NOT EXISTS

    FOR (w:Water)

    REQUIRE w.water_id IS UNIQUE
    """,
    database_=DATABASE
)

print("Water constraint created")

Water constraint created


เพิ่ม customer หลายๆคน

In [ ]:
customers = [
    {"customer_id": "C001", "name": "Kanya"},
    {"customer_id": "C002", "name": "Nattapong"},
    {"customer_id": "C003", "name": "Siriporn"},
    {"customer_id": "C004", "name": "Thanakrit"},
    {"customer_id": "C005", "name": "Preecha"},
    {"customer_id": "C006", "name": "Malee"},
    {"customer_id": "C007", "name": "Chaiyut"},
    {"customer_id": "C008", "name": "Nichada"},
    {"customer_id": "C009", "name": "Somsak"},
    {"customer_id": "C010", "name": "Ratree"}
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $customers AS row

    MERGE (u:Customer {
        customer_id: row.customer_id
    })

    SET
        u.name = row.name

    RETURN count(u) AS total
    """,

    customers=customers,
    database_=DATABASE
)

print(
    "จำนวน Customer ที่ประมวลผล =",
    result.records[0]["total"]
)

จำนวน Customer ที่ประมวลผล = 10


In [ ]:
waters = [
    {
        "water_id": "W101",
        "name": "Singha"
    },
    {
        "water_id": "W102",
        "name": "Crystal"
    },
    {
        "water_id": "W103",
        "name": "Minere"
    },
    {
        "water_id": "W104",
        "name": "Nestle Pure Life"
    },
    {
        "water_id": "W105",
        "name": "Perrier"
    },
    {
        "water_id": "W106",
        "name": "Namthip"
    },
    {
        "water_id": "W107",
        "name": "evian"
    }
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $waters AS row

    MERGE (w:Water {
        water_id: row.water_id
    })

    SET w.name = row.name

    RETURN count(w) AS total
    """,

    waters=waters,
    database_=DATABASE
)

print(
    "จำนวน Water ที่ประมวลผล =",
    result.records[0]["total"]
)

จำนวน Water ที่ประมวลผล = 7


ดู customer และ water ทั้งหมด

In [ ]:
result = driver.execute_query(
    """
    MATCH (n)

    WHERE n:Customer OR n:Water

    RETURN
        labels(n) AS type,
        properties(n) AS data
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

{'type': ['Customer'], 'data': {'name': 'Kanya', 'customer_id': 'C001'}}
{'type': ['Customer'], 'data': {'name': 'Nattapong', 'customer_id': 'C002'}}
{'type': ['Customer'], 'data': {'name': 'Siriporn', 'customer_id': 'C003'}}
{'type': ['Customer'], 'data': {'name': 'Thanakrit', 'customer_id': 'C004'}}
{'type': ['Customer'], 'data': {'name': 'Preecha', 'customer_id': 'C005'}}
{'type': ['Customer'], 'data': {'name': 'Malee', 'customer_id': 'C006'}}
{'type': ['Customer'], 'data': {'name': 'Chaiyut', 'customer_id': 'C007'}}
{'type': ['Customer'], 'data': {'name': 'Nichada', 'customer_id': 'C008'}}
{'type': ['Customer'], 'data': {'name': 'Somsak', 'customer_id': 'C009'}}
{'type': ['Customer'], 'data': {'name': 'Ratree', 'customer_id': 'C010'}}
{'type': ['Water'], 'data': {'name': 'Singha', 'water_id': 'W101'}}
{'type': ['Water'], 'data': {'name': 'Crystal', 'water_id': 'W102'}}
{'type': ['Water'], 'data': {'name': 'Minere', 'water_id': 'W103'}}
{'type': ['Water'], 'data': {'name': 'Nestle P

Step 10 — สร้าง Relationship SIMILAR_TO

กำหนดความสัมพันธ์ของผู้ใช้ที่มีรสนิยมการเลือกน้ำแร่คล้ายกัน

In [ ]:
similarities = [
    {"customer1": "C001", "customer2": "C002"},
    {"customer1": "C001", "customer2": "C005"},
    {"customer1": "C001", "customer2": "C008"},
    {"customer1": "C002", "customer2": "C008"},
    {"customer1": "C002", "customer2": "C009"},
    {"customer1": "C003", "customer2": "C007"},
    {"customer1": "C003", "customer2": "C010"},
    {"customer1": "C004", "customer2": "C009"},
    {"customer1": "C005", "customer2": "C006"},
    {"customer1": "C006", "customer2": "C009"},
    {"customer1": "C007", "customer2": "C010"},
    {"customer1": "C004", "customer2": "C005"}
]

In [ ]:
driver.execute_query(
    """
    MATCH
        (a:Customer {customer_id: "C001"}),
        (b:Customer {customer_id: "C002"})

    MERGE
        (a)-[:SIMILAR_TO]->(b)
    """,
    database_=DATABASE
)

print("สร้าง SIMILAR_TO สำเร็จ")

สร้าง SIMILAR_TO สำเร็จ


In [ ]:
driver.execute_query(
    """
    MATCH
        (a:Customer {customer_id: "C001"}),
        (b:Customer {customer_id: "C005"})

    MERGE
        (a)-[:SIMILAR_TO]->(b)
    """,
    database_=DATABASE
)

print("สร้าง SIMILAR_TO สำเร็จ")

สร้าง SIMILAR_TO สำเร็จ


In [ ]:
driver.execute_query(
    """
    MATCH
        (a:Customer {customer_id: "C001"}),
        (b:Customer {customer_id: "C008"})

    MERGE
        (a)-[:SIMILAR_TO]->(b)
    """,
    database_=DATABASE
)

print("สร้าง SIMILAR_TO สำเร็จ")

สร้าง SIMILAR_TO สำเร็จ


In [ ]:
driver.execute_query(
    """
    MATCH
        (a:Customer {customer_id: "C002"}),
        (b:Customer {customer_id: "C008"})

    MERGE
        (a)-[:SIMILAR_TO]->(b)
    """,
    database_=DATABASE
)

print("สร้าง SIMILAR_TO สำเร็จ")

สร้าง SIMILAR_TO สำเร็จ


In [ ]:
driver.execute_query(
    """
    MATCH
        (a:Customer {customer_id: "C002"}),
        (b:Customer {customer_id: "C009"})

    MERGE
        (a)-[:SIMILAR_TO]->(b)
    """,
    database_=DATABASE
)

print("สร้าง SIMILAR_TO สำเร็จ")

สร้าง SIMILAR_TO สำเร็จ


In [ ]:
driver.execute_query(
    """
    MATCH
        (a:Customer {customer_id: "C003"}),
        (b:Customer {customer_id: "C007"})

    MERGE
        (a)-[:SIMILAR_TO]->(b)
    """,
    database_=DATABASE
)

print("สร้าง SIMILAR_TO สำเร็จ")

สร้าง SIMILAR_TO สำเร็จ


In [ ]:
driver.execute_query(
    """
    MATCH
        (a:Customer {customer_id: "C003"}),
        (b:Customer {customer_id: "C010"})

    MERGE
        (a)-[:SIMILAR_TO]->(b)
    """,
    database_=DATABASE
)

print("สร้าง SIMILAR_TO สำเร็จ")

สร้าง SIMILAR_TO สำเร็จ


Step 11 — Query ความสัมพันธ์ คนที่สัมพันธ์กับ C002

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:Customer {customer_id:'C002'})
        -[:SIMILAR_TO]->
        (similar:Customer)

    RETURN similar.name AS similar
    """,
    database_=DATABASE
)

for record in result.records:
    print(record["similar"])

Nichada
Somsak


Step 12 — สร้าง LIKES Relationship

กำหนดประวัติการชอบน้ำแร่

In [ ]:
likes = [
    {"customer_id": "C001", "water_id": "W101"},
    {"customer_id": "C001", "water_id": "W102"},
    {"customer_id": "C002", "water_id": "W101"},
    {"customer_id": "C002", "water_id": "W102"},
    {"customer_id": "C002", "water_id": "W104"},
    {"customer_id": "C003", "water_id": "W101"},
    {"customer_id": "C003", "water_id": "W103"},
    {"customer_id": "C004", "water_id": "W104"},
    {"customer_id": "C004", "water_id": "W105"},
    {"customer_id": "C005", "water_id": "W102"},
    {"customer_id": "C005", "water_id": "W106"},
    {"customer_id": "C006", "water_id": "W101"},
    {"customer_id": "C006", "water_id": "W106"},
    {"customer_id": "C006", "water_id": "W107"},
    {"customer_id": "C007", "water_id": "W103"},
    {"customer_id": "C007", "water_id": "W105"},
    {"customer_id": "C008", "water_id": "W101"},
    {"customer_id": "C008", "water_id": "W102"},
    {"customer_id": "C008", "water_id": "W107"},
    {"customer_id": "C009", "water_id": "W104"},
    {"customer_id": "C009", "water_id": "W106"},
    {"customer_id": "C010", "water_id": "W103"},
    {"customer_id": "C010", "water_id": "W105"},
    {"customer_id": "C010", "water_id": "W107"}
]

สร้าง Relationship

In [ ]:
driver.execute_query(
    """
    UNWIND $likes AS row

    MATCH
        (u:Customer {customer_id: row.customer_id}),
        (w:Water {water_id: row.water_id})

    MERGE
        (u)-[r:LIKES]->(w)
    """,

    likes=likes,
    database_=DATABASE
)

print("สร้าง LIKES สำเร็จ")

สร้าง LIKES สำเร็จ


Step 13 — ดูว่าใครชอบน้ำแร่อะไร

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:Customer)
        -[r:LIKES]->
        (w:Water)

    RETURN
        u.name AS customer,
        w.name AS water

    ORDER BY customer, water
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["customer"],
        "->",
        record["water"]
    )

Chaiyut -> Minere
Chaiyut -> Perrier
Kanya -> Crystal
Kanya -> Singha
Malee -> Namthip
Malee -> Singha
Malee -> evian
Nattapong -> Crystal
Nattapong -> Nestle Pure Life
Nattapong -> Singha
Nichada -> Crystal
Nichada -> Singha
Nichada -> evian
Preecha -> Crystal
Preecha -> Namthip
Ratree -> Minere
Ratree -> Perrier
Ratree -> evian
Siriporn -> Minere
Siriporn -> Singha
Somsak -> Namthip
Somsak -> Nestle Pure Life
Thanakrit -> Nestle Pure Life
Thanakrit -> Perrier


Step 14 — นำผลลัพธ์มาแสดงเป็นตาราง Pandas

In [ ]:
import pandas as pd

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:Customer)
        -[r:LIKES]->
        (w:Water)

    RETURN
        u.customer_id AS customer_id,
        u.name AS customer,
        w.water_id AS water_id,
        w.name AS water

    ORDER BY customer_id
    """,
    database_=DATABASE
)

data = [
    record.data()
    for record in result.records
]

df = pd.DataFrame(data)

df

,customer_id,customer,water_id,water
0,C001,Kanya,W101,Singha
1,C001,Kanya,W102,Crystal
2,C002,Nattapong,W101,Singha
3,C002,Nattapong,W102,Crystal
4,C002,Nattapong,W104,Nestle Pure Life
5,C003,Siriporn,W101,Singha
6,C003,Siriporn,W103,Minere
7,C004,Thanakrit,W104,Nestle Pure Life
8,C004,Thanakrit,W105,Perrier
9,C005,Preecha,W102,Crystal


Step 15 — Traversal หลายทอด

โจทย์:

น้ำแร่ที่ผู้ใช้ที่มีความสนใจคล้ายกับ Kanya ชอบ มีอะไรบ้าง?

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (me:Customer {customer_id:'C001'})
        -[:SIMILAR_TO]->
        (similar:Customer)
        -[:LIKES]->
        (water:Water)

    RETURN
        me.name AS me,
        similar.name AS similar,
        water.name AS water

    ORDER BY similar, water
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["me"],
        "-> ผู้ใช้คล้าย:",
        record["similar"],
        "-> ชอบ:",
        record["water"]
    )

Kanya -> ผู้ใช้คล้าย: Nattapong -> ชอบ: Crystal
Kanya -> ผู้ใช้คล้าย: Nattapong -> ชอบ: Nestle Pure Life
Kanya -> ผู้ใช้คล้าย: Nattapong -> ชอบ: Singha
Kanya -> ผู้ใช้คล้าย: Nichada -> ชอบ: Crystal
Kanya -> ผู้ใช้คล้าย: Nichada -> ชอบ: Singha
Kanya -> ผู้ใช้คล้าย: Nichada -> ชอบ: evian
Kanya -> ผู้ใช้คล้าย: Preecha -> ชอบ: Crystal
Kanya -> ผู้ใช้คล้าย: Preecha -> ชอบ: Namthip


Step 16 — Aggregation

โจทย์:

ลูกค้าแต่ละคนชอบน้ำแร่กี่แบรนด์?

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:Customer)
        -[:LIKES]->
        (w:Water)

    RETURN
        u.name AS customer,
        count(w) AS total_waters

    ORDER BY total_waters DESC
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["customer"],
        "=",
        record["total_waters"],
        "แบรนด์"
    )

Nattapong = 3 แบรนด์
Malee = 3 แบรนด์
Nichada = 3 แบรนด์
Ratree = 3 แบรนด์
Kanya = 2 แบรนด์
Siriporn = 2 แบรนด์
Preecha = 2 แบรนด์
Chaiyut = 2 แบรนด์
Thanakrit = 2 แบรนด์
Somsak = 2 แบรนด์


Step 17 — น้ำแร่ยี่ห้อไหนได้รับความนิยมมากที่สุด

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (:Customer)
        -[:LIKES]->
        (w:Water)

    RETURN
        w.name AS water,
        count(*) AS like_count

    ORDER BY like_count DESC

    """,
    database_=DATABASE
)

pd.DataFrame(
    [record.data() for record in result.records]
)

,water,like_count
0,Singha,5
1,Crystal,4
2,Minere,3
3,Nestle Pure Life,3
4,Perrier,3
5,Namthip,3
6,evian,3


Step 18 — Mini Project: Recommendation System

โจทย์สำคัญที่สุด

แนะนำน้ำแร่ให้ Kanya จากน้ำแร่ที่ผู้ใช้ที่มีความสนใจคล้ายกันเคยชอบ แต่ Kanya ยังไม่เคยชอบ

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (me:Customer {customer_id:$customer_id})
        -[:SIMILAR_TO]->
        (similar:Customer)
        -[:LIKES]->
        (water:Water)

    WHERE NOT EXISTS {

        MATCH
            (me)-[:LIKES]->(water)

    }

    RETURN
        water.water_id AS water_id,
        water.name AS recommendation,
        count(DISTINCT similar) AS similar_score

    ORDER BY
        similar_score DESC,
        recommendation
    """,

    customer_id="C001",
    database_=DATABASE
)

recommend_df = pd.DataFrame(
    [record.data() for record in result.records]
)

recommend_df

,water_id,recommendation,similar_score
0,W106,Namthip,1
1,W104,Nestle Pure Life,1
2,W107,evian,1


ทำไมคำแนะนำจึงได้คะแนนสูงกว่า?

เพราะระบบนับจำนวนผู้ใช้ที่มีความสัมพันธ์ `SIMILAR_TO` กับผู้ใช้เป้าหมาย และชอบน้ำแร่รายการเดียวกัน

ดังนั้นน้ำแร่ที่มีผู้ใช้คล้ายกันชอบหลายคน จะได้คะแนนสูงกว่า

Step 19 — ทำเป็น Python Function

เปลี่ยน Mini Project ให้ดูเหมือนโปรแกรมจริง

In [ ]:
def recommend_waters(customer_id):

    query = """
    MATCH
        (me:Customer {customer_id:$customer_id})
        -[:SIMILAR_TO]->
        (similar:Customer)
        -[:LIKES]->
        (water:Water)

    WHERE NOT EXISTS {

        MATCH
            (me)-[:LIKES]->(water)

    }

    RETURN
        water.water_id AS water_id,
        water.name AS recommendation,
        count(DISTINCT similar) AS score

    ORDER BY
        score DESC,
        recommendation
    """

    result = driver.execute_query(
        query,
        customer_id=customer_id,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records]
    )

In [ ]:
recommend_waters("C001")

,water_id,recommendation,score
0,W106,Namthip,1
1,W104,Nestle Pure Life,1
2,W107,evian,1


Step 20 — Function ค้นหา Customer

In [ ]:
def find_customer(customer_id):

    result = driver.execute_query(
        """
        MATCH (u:Customer {customer_id:$customer_id})

        RETURN
            u.customer_id AS id,
            u.name AS name
        """,

        customer_id=customer_id,
        database_=DATABASE
    )

    if len(result.records) == 0:
        return "ไม่พบ Customer"

    return result.records[0].data()

In [ ]:
find_customer("C003")

{'id': 'C003', 'name': 'Siriporn'}

Step 21 — Function ดูน้ำแร่ที่ Customer เคยชอบ

In [ ]:
def liked_waters(customer_id):

    result = driver.execute_query(
        """
        MATCH
            (u:Customer {customer_id:$customer_id})
            -[r:LIKES]->
            (w:Water)

        RETURN
            w.water_id AS water_id,
            w.name AS name

        ORDER BY name
        """,

        customer_id=customer_id,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records]
    )

In [ ]:
liked_waters("C005")

,water_id,name
0,W102,Crystal
1,W106,Namthip


Step 22 — ดู Graph ใน Neo4j Aura

หลังจากรันข้อมูลจาก Colab แล้ว ให้นักเรียนกลับไปที่ Neo4j Aura → Query

รัน

In [ ]:
#Step 23 — ตรวจจำนวน Node และ Relationship
result = driver.execute_query(
    """
    MATCH (n)

    OPTIONAL MATCH (n)-[r]->()

    RETURN
        count(DISTINCT n) AS nodes,
        count(r) AS relationships
    """,
    database_=DATABASE
)

print(result.records[0].data())

{'nodes': 17, 'relationships': 31}


In [ ]:
#Step 24 — ปิด Connection เมื่อเรียนเสร็จ

driver.close()

print("Neo4j connection closed")

Neo4j connection closed
